# 2.4 Merge panel data with equipment and FC calculator data

This notebook does the following:
    - Merges the dataset with the equipment calculators

## Set-up

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
import re
from pathlib import Path
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
import os
from fill_missing_mode import fill_with_equipment_mode
from assign_set_temp import assign_set_temp

In [2]:
# Load data
equipment = pd.read_csv(
    config.PROCESSED_DATA / "panel_processed_4.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

## (1) Energy use calculations

In [3]:
# Initialise output columns; formulas are applied per equipment type below
HOURS_PER_YEAR = 24 * 365   # 8760

equipment["electricity_use_kwh_year"] = np.nan
equipment["gas_use_kwh_year"] = np.nan

### (3a) Always-on equipment (Category 1)

In [4]:
# Fridges
# Annual = 365 × Baseline × (1 + Minutes_Open_Per_Day × Door_Opening_Penalty) 
# Notes: (1) we assume number of openings = minutes open
#        (2) we assume baseline is per day
mask = equipment["equipment"] == "fridge"
door_term = equipment["door_openings"] * equipment["door_opening_penalty_xbl_xminutes_open"]

equipment.loc[mask, "electricity_use_kwh_year"] = (
    365 * 24
    * equipment.loc[mask, "electricity_use_kwh_per_hour"]
    * (1 + door_term[mask])
)

In [5]:
# Freezers
# Annual = Hourly_Baseline × Ice_Penalty × 365 × HFC_Penalty × (24 + Door_Term)
# Door_Term: upright freezers without plastic drawers get an extra No_Drawers_Penalty multiplier
# Notes: (1) we assume number of openings = minutes open

mask = equipment["equipment"] == "freezer"

no_drawers_applies = (
    (equipment["size_freezer_1"] == "Upright") &
    (equipment["drawers"] == "No there are no drawers or most are missing")
)
door_term = np.where(
    no_drawers_applies,
    equipment["door_openings"] * 
        equipment["no_drawers_penalty_xblx_minutes_open"] * 
        equipment["door_opening_penalty_xbl_xminutes_open"],
    equipment["door_openings"] * 
        equipment["door_opening_penalty_xbl_xminutes_open"],
)
ice_factor = np.where(equipment["icing"] == "Yes, iced up", equipment["ice_penalty_xbl"], 1.0)
hfc_factor = np.where(equipment["refrigerant"].str.contains("HFCs", na=False), equipment["hfc_not_hc_coolant_penalty_xbl"], 1.0)

equipment.loc[mask, "electricity_use_kwh_year"] = (
    equipment.loc[mask, "electricity_use_kwh_per_hour"]
    * ice_factor[mask]
    * hfc_factor[mask]
    * 365
    * (24 + door_term[mask])
)

In [6]:
# ULT Freezers
# Annual = 365 × (Baseline × Condition Factors) × (1 + Minutes_Open × Door_Opening_Penalty)
# Condition Factors: penalties for seals, spacing, and filter are composite - need to use specific ones
# Notes: (1) we assume number of openings = minutes open
#        (2) we assume baseline is per day

mask = equipment["equipment"] == "ult"

# Create masks for each condition and their combinations
seals_mask = (equipment["damaged_seals_penalty"] == 1)
spacing_mask = (equipment["no_spacing_penalty"] == 1)
filter_mask = (equipment["clogged_filter_penalty"] == 1)

all_conditions_mask = seals_mask & spacing_mask & filter_mask

only_seals_mask = seals_mask & ~spacing_mask & ~filter_mask
only_spacing_mask = ~seals_mask & spacing_mask & ~filter_mask
only_filter_mask = ~seals_mask & ~spacing_mask & filter_mask

no_seals_mask = ~seals_mask & spacing_mask & filter_mask
no_spacing_mask = seals_mask & ~spacing_mask & filter_mask
no_filter_mask = seals_mask & spacing_mask & ~filter_mask

# Start with default factor of 1.0, then apply specific factors for each condition combination
condition_factor = 1.0

# Only one condition 
condition_factor = np.where(only_seals_mask, equipment["damaged_seals_penalty_xbl"], condition_factor)
condition_factor = np.where(only_spacing_mask, equipment["no_spacing_penalty_xbl"], condition_factor)
condition_factor = np.where(only_filter_mask, equipment["clogged_filters_penalty_xbl"], condition_factor)

# Only two conditions (need to use specific combined factors)
condition_factor = np.where(no_seals_mask, equipment["no_spacing_and_clogged_filters_penalty_xbl"], condition_factor)
condition_factor = np.where(no_spacing_mask, equipment["damaged_seals_and_clogged_filters_penalty"], condition_factor)
condition_factor = np.where(no_filter_mask, equipment["damaged_seals_and_no_spacing_penalty_xbl"], condition_factor)

# All conditions
condition_factor = np.where(all_conditions_mask, equipment["no_spacing_and_clogged_filters_and_damaged_seal_penalty_xbl"], condition_factor)

door_term = equipment["door_openings"] * equipment["door_opening_penalty_xbl_xminutes_open"]

equipment.loc[mask, "electricity_use_kwh_year"] = (
    365 * 24
    * equipment.loc[mask, "electricity_use_kwh_per_hour"]
    * condition_factor[mask]
    * (1 + door_term[mask])
)

In [7]:
# CO2 Incubators (always on, no penalties)
# Annual = 365 × 24 × Hourly_Energy_Use
mask = equipment["equipment"] == "incubator"

equipment.loc[mask, "electricity_use_kwh_year"] = (
    HOURS_PER_YEAR * equipment.loc[mask, "electricity_use_kwh_per_hour"]
)

### (1b) Usage-based equipment (water baths, heat blocks, IT equipment, cryostats, drying cabinets, MSC, FCs)

In [8]:
# Water Baths
# Annual = Hours × Days × Hourly × Lid_Off_Penalty × Bead_Penalty
mask = equipment["equipment"] == "bath"

lid_factor  = np.where(equipment["lid_off_penalty"] == 1, equipment["lid_off_penalty_xbl"].fillna(1.0), 1.0)
bead_factor = np.where(equipment["beads_penalty"]   == 1, equipment["beads_not_water_penalty_xbl"].fillna(1.0), 1.0)

equipment.loc[mask, "electricity_use_kwh_year"] = (
    equipment.loc[mask, "hours"]
    * equipment.loc[mask, "days"]
    * equipment.loc[mask, "electricity_use_kwh_per_hour"]
    * lid_factor[mask]
    * bead_factor[mask]
)

In [9]:
# Heat Blocks and IT Equipment (no penalties)
# Annual = Hours × Days × Hourly_Energy_Use
for eq_type in ["heater", "it"]:
    mask = equipment["equipment"] == eq_type
    equipment.loc[mask, "electricity_use_kwh_year"] = (
        equipment.loc[mask, "hours"]
        * equipment.loc[mask, "days"]
        * equipment.loc[mask, "electricity_use_kwh_per_hour"]
    )

In [10]:
# Cryostats
# No energy-saving mode:  Annual = 8760 × Hourly
# With energy-saving mode: Annual = (Hours × Days × Hourly)
#                                 + ((8760 - Hours × Days) × Hourly × Standby_Penalty)
mask        = equipment["equipment"] == "cryostat"
no_standby  = mask & (equipment["sleep_mode"] == "No Energy Saving Mode Available")
has_standby = mask & (equipment["sleep_mode"] == "Energy Saving Mode Available")

usage_hours = equipment["hours"] * equipment["days"]

equipment.loc[no_standby, "electricity_use_kwh_year"] = (
    HOURS_PER_YEAR * equipment.loc[no_standby, "electricity_use_kwh_per_hour"]
)
equipment.loc[has_standby, "electricity_use_kwh_year"] = (
    usage_hours[has_standby] * equipment.loc[has_standby, "electricity_use_kwh_per_hour"]
    + (HOURS_PER_YEAR - usage_hours[has_standby])
    * equipment.loc[has_standby, "electricity_use_kwh_per_hour"]
    * equipment.loc[has_standby, "standby_penalty"]
)

In [11]:
# Drying Cabinets (Glassware)
# Annual = Hours × Days × Hourly × Fan_Penalty_Multiplier
mask = equipment["equipment"] == "glassware"

fan_factor = np.where(equipment["fan_penalty"] == 1, equipment["fan_penalty_xbl"].fillna(1.0), 1.0)

equipment.loc[mask, "electricity_use_kwh_year"] = (
    equipment.loc[mask, "hours"]
    * equipment.loc[mask, "days"]
    * equipment.loc[mask, "electricity_use_kwh_per_hour"]
    * fan_factor[mask]
)

In [12]:
# Microbial Safety Cabinets
# Electricity: Hours × Days × Hourly × Ducted_Electricity_Penalty (penalty = 1 if not ducted)
# Gas (ducted only): Hours × Days × Ducted_Gas_Use_Per_Hour
mask     = equipment["equipment"] == "microbio"
is_ducted = equipment["ducting_penalty"] == 1
ducting_factor = np.where(is_ducted, equipment["ducted_electricity_penalty_xbl"], 1.0)

equipment.loc[mask, "electricity_use_kwh_year"] = (
    equipment.loc[mask, "hours"]
    * equipment.loc[mask, "days"]
    * equipment.loc[mask, "electricity_use_kwh_per_hour"]
    * ducting_factor[mask]
)
equipment.loc[mask, "gas_use_kwh_year"] = np.where(
    is_ducted[mask],
    equipment.loc[mask, "hours"] 
    * equipment.loc[mask, "days"] 
    * equipment.loc[mask, "ducted_gas_penalty_additional_kwh_per_hour"],
    0.0,
)

In [13]:
# Print the equipment types where we are missing electricity use values (should be none)
missing_electricity = equipment.loc[
    equipment["electricity_use_kwh_year"].isna(), "equipment"
].unique()
print("Equipment types with missing electricity use values:", missing_electricity)

# Print the equipment types where electricity = 0 - need to fix these
zero_electricity = equipment.loc[
    equipment["electricity_use_kwh_year"] == 0, "equipment"
].unique()
print("Equipment types with zero electricity use:", zero_electricity)

# Print the equipment types where electricity per hour = 0 
zero_electricity_per_hour = equipment.loc[
    equipment["electricity_use_kwh_per_hour"] == 0, "equipment"
].unique()
print("Equipment types with zero electricity use per hour:", zero_electricity_per_hour)

# print the equipment types where electricity is 0 and hours/days are > 0 - these are the ones we need to fix
zero_electricity_but_used = equipment.loc[
    (equipment["electricity_use_kwh_year"] == 0) &
    (equipment["hours"] > 0) &
    (equipment["days"] > 0),
    "equipment"
].unique()
print("Equipment types with zero electricity use but have hours and days > 0:", zero_electricity_but_used)

# Create indicator for not on (hours or days = 0)
equipment["not_on"] = (equipment["hours"] == 0) | (equipment["days"] == 0)

Equipment types with missing electricity use values: ['fc']
Equipment types with zero electricity use: ['heater' 'microbio' 'glassware' 'it']
Equipment types with zero electricity use per hour: []
Equipment types with zero electricity use but have hours and days > 0: []


### (3c) Fume cupboards

In [14]:
# Fume Cupboards — shared geometry
# Sash height: 0.5 m (fixed); sash width: survey_width - 0.3 m; tested velocity: survey_velocity × 1.075
ELEC_PER_M3 = config.ELEC_PER_M3  # kWh/m³
GAS_PER_M3  = config.GAS_PER_M3     # kWh/m³

fc_mask    = equipment["equipment"] == "fc"
sash_width = equipment["sash_width"] - 0.3        # m
face_vel   = equipment["face_velocity_m/s"] * 1.075  # m/s (tested)
Q          = face_vel * (0.5 * sash_width)           # m³/s

# CAV: constant flow 24/7
cav_mask   = fc_mask & (equipment["controller_type"] == "CAV")
cav_volume = Q * 3600 * HOURS_PER_YEAR              # m³/year

equipment.loc[cav_mask, "electricity_use_kwh_year"] = cav_volume[cav_mask] * ELEC_PER_M3
equipment.loc[cav_mask, "gas_use_kwh_year"]         = cav_volume[cav_mask] * GAS_PER_M3

# VAV: variable flow — open at design volume, closed at 25% flow
# Loading_Factor = surface × 1.1 × (0 if contents raised, else 1)
vav_mask = fc_mask & (equipment["controller_type"] == "VAV")

loading_factor = (
    equipment["surface"] * 1.1
    * np.where(equipment["lifted"] == "Yes", 0, 1)
)
open_hours   = equipment["hours_open"] * equipment["days"]
closed_hours = HOURS_PER_YEAR - open_hours

vav_volume = (
        (open_hours * Q * (1 + loading_factor))
        + (closed_hours * Q * 0.25)
    ) * 3600

equipment.loc[vav_mask, "electricity_use_kwh_year"] = vav_volume[vav_mask] * ELEC_PER_M3
equipment.loc[vav_mask, "gas_use_kwh_year"]         = vav_volume[vav_mask] * GAS_PER_M3

# Check that we have no missing values for electricity use for fume cupboards after applying formulas
missing_fc_electricity = equipment.loc[
    (equipment["equipment"] == "fc") 
    & (equipment["electricity_use_kwh_year"].isna())
]
print("Number of fume cupboards with missing electricity use values:", missing_fc_electricity.shape[0])

Number of fume cupboards with missing electricity use values: 0


In [15]:
# Check that electricity_use_kwh_year is not missing for any equipment types
missing_electricity = equipment.loc[
    equipment["electricity_use_kwh_year"].isna(), "equipment"
].unique()
print("Equipment types with missing electricity use values:", missing_electricity)

Equipment types with missing electricity use values: []


### (3d) Compute totals (multiply by number of units)

In [16]:
# Rename electricity_use_kwh_year to electricity_use_kwh_year_per_unit to make it clear that this is per unit, not total
equipment.rename(columns={
    "electricity_use_kwh_year": "electricity_use_kwh_year_per_unit", 
    "gas_use_kwh_year": "gas_use_kwh_year_per_unit"
}, inplace=True)


# Create variables for totals (multiply per unit use by number of units)
equipment["electricity_use_kwh_year"] = equipment["electricity_use_kwh_year_per_unit"] * equipment["number"]
equipment["gas_use_kwh_year"] = equipment["gas_use_kwh_year_per_unit"] * equipment["number"]
equipment["total_energy_kwh_year"]     = (
    equipment["electricity_use_kwh_year"] + equipment["gas_use_kwh_year"]
)

In [17]:
# Order cols
columns_to_keep = [
    "labgroupid", "survey", "equipment", "type_no",
    "number", "electricity_use_kwh_year_per_unit", "electricity_use_kwh_year",
    "hours", "days", "not_on"
]

other_cols = [col for col in equipment.columns if col not in columns_to_keep]

equipment = equipment[columns_to_keep + other_cols]

## (4) Checking the data

In [18]:
# Check the distribution of energy use per unit across equipment types to identify any outliers
energy_per_unit_by_equipment = equipment.groupby("equipment")["electricity_use_kwh_year_per_unit"].median().sort_values(ascending=False)
print("Median annual electricity use per unit (kWh) by equipment type:")
print(energy_per_unit_by_equipment)

Median annual electricity use per unit (kWh) by equipment type:
equipment
cryostat     5663.340000
ult          3966.403614
fc           1395.703922
incubator     471.579708
fridge        370.475292
freezer       359.803491
microbio      321.300000
glassware     207.187914
it            179.520000
bath           52.785000
heater          5.033340
Name: electricity_use_kwh_year_per_unit, dtype: float64


In [19]:
# Verify all equipment has an annual electricity value
missing_total = equipment[equipment["electricity_use_kwh_year"].isna()]
if missing_total.empty:
    print("All equipment has a total annual electricity value.")
else:
    print(f"{len(missing_total)} rows missing electricity_use_kwh_year:")
    print(missing_total.groupby("equipment").size())

# Verify all equipment has an annual electricity value per unit
missing_per_unit = equipment[equipment["electricity_use_kwh_year_per_unit"].isna()]
if missing_per_unit.empty:
    print("All equipment has an annual electricity value per unit.")
else:
    print(f"{len(missing_per_unit)} rows missing electricity_use_kwh_year_per_unit:")
    print(missing_per_unit.groupby("equipment").size())

All equipment has a total annual electricity value.
All equipment has an annual electricity value per unit.


In [20]:
# Check possible ranges of electricity use per unit for ULTs
max_ult_baseline = equipment.loc[equipment["equipment"] == "ult", "electricity_use_kwh_per_hour"].max()
max_ult_condition_factor = equipment.loc[equipment["equipment"] == "ult", "no_spacing_and_clogged_filters_and_damaged_seal_penalty_xbl"].max()
max_door_openings = equipment.loc[equipment["equipment"] == "ult", "door_openings"].max()
max_door_opening_penalty = equipment.loc[equipment["equipment"] == "ult", "door_opening_penalty_xbl_xminutes_open"].max()
max_ult_energy = 365 * 24 * max_ult_baseline * max_ult_condition_factor * (1 + max_door_openings * max_door_opening_penalty)
print(f"Max possible ULT electricity: {max_ult_energy:.2f} kWh/year")
max_ult_observed  = equipment.loc[equipment["equipment"] == "ult", "electricity_use_kwh_year_per_unit"].max()
print(f"Max observed ULT electricity per unit: {max_ult_observed:.2f} kWh/year")

Max possible ULT electricity: 38110.00 kWh/year
Max observed ULT electricity per unit: 14731.05 kWh/year


In [21]:
# Check possible ranges of electricity use per unit for FCs
max_fc_sash_width = equipment.loc[equipment["equipment"] == "fc", "sash_width"].max()
max_fc_face_velocity = equipment.loc[equipment["equipment"] == "fc", "face_velocity_m/s"].max()
max_fc_Q = max_fc_face_velocity * 0.5 * (max_fc_sash_width - 0.3)
max_fc_energy = max_fc_Q * 3600 * HOURS_PER_YEAR * ELEC_PER_M3
print(f"Max possible FC electricity: {max_fc_energy:.2f} kWh/year")
max_fc_observed  = equipment.loc[equipment["equipment"] == "fc", "electricity_use_kwh_year_per_unit"].max()
print(f"Max observed FC electricity per unit: {max_fc_observed:.2f} kWh/year")

Max possible FC electricity: 25762.88 kWh/year
Max observed FC electricity per unit: 11078.04 kWh/year


In [22]:
# Check possible ranges of electricity use per unit for cryostats
max_cryostat_baseline = equipment.loc[equipment["equipment"] == "cryostat", "electricity_use_kwh_per_hour"].max()
max_cryostat_energy = 365 * 24 * max_cryostat_baseline
print(f"Max possible cryostat electricity: {max_cryostat_energy:.2f} kWh/year")
max_cryostat_observed  = equipment.loc[equipment["equipment"] == "cryostat", "electricity_use_kwh_year_per_unit"].max()
print(f"Max observed cryostat electricity per unit: {max_cryostat_observed:.2f} kWh/year")

min_cryostat_baseline = equipment.loc[equipment["equipment"] == "cryostat", "electricity_use_kwh_per_hour"].min()
min_cryostat_always_standby = 365 * 24 * min_cryostat_baseline * 0.4
print(f"Min possible cryostat electricity if always on standby: {min_cryostat_always_standby:.2f} kWh/year")
min_cryostat_observed = equipment.loc[equipment["equipment"] == "cryostat", "electricity_use_kwh_year_per_unit"].min()
print(f"Min observed cryostat electricity per unit: {min_cryostat_observed:.2f} kWh/year")

Max possible cryostat electricity: 6670.74 kWh/year
Max observed cryostat electricity per unit: 6670.74 kWh/year
Min possible cryostat electricity if always on standby: 2265.34 kWh/year
Min observed cryostat electricity per unit: 2457.20 kWh/year


## Clean up and save processed data

In [23]:
# Save processed data
equipment.to_csv(config.PROCESSED_DATA / "panel_processed_5.csv", index =False)